# Physics-Augmented AI-Image Detection — Notebook 01: data, bias audit & features

This notebook turns a GenImage-style dataset into cached features that every later notebook trains on:

| Output (in `/kaggle/working`) | What it is |
|---|---|
| `manifest.csv` | one row per image: path, label (0 real / 1 fake), generator, split, format, size, JPEG quality, md5 |
| `features/geometry.parquet` | 18 projective-geometry **physics** features per image (vanishing-point convergence, camera consistency) |
| `features/clip_vitl14.npy` | CLIP ViT-L/14 image embeddings (the UnivFD-style **baseline**), row-aligned with the manifest |
| `physfeat/` | the feature-extraction package, reused by notebooks 02 and 03 |
| `summary.json` | dataset stats, bias-audit result and the preview cross-generator table |

**Kaggle setup (once):**
1. *Add Input* → search **"Unbiased Tiny GenImage"** (`cartografia/unbiased-tiny-genimage`). Fallback: `yangsangtai/tiny-genimage` — the notebook still works, and the bias audit will tell you how biased it is.
2. *Settings* → **Accelerator: GPU T4** (only the CLIP cell uses it) and **Internet: On** (to download CLIP; needs a phone-verified account).
3. **Save Version → Save & Run All (Commit)** rather than running interactively — commits survive the idle timeout. Every heavy step is chunked and resumable.

Runtime on ~40k images: metadata ≈ 3 min, geometry ≈ 10–15 min (4 CPU cores), CLIP ≈ 5–8 min (T4).

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle/input")
DATA_ROOTS = [p for p in os.environ.get("PHYSFAKE_DATA_ROOTS", "/kaggle/input").split(os.pathsep) if p]
OUT_DIR = Path(os.environ.get("PHYSFAKE_OUT", "/kaggle/working" if IN_KAGGLE else "./work")).resolve()
CLIP_MODEL = os.environ.get("PHYSFAKE_CLIP_MODEL", "openai/clip-vit-large-patch14")

MAX_PER_GROUP = int(os.environ.get("PHYSFAKE_MAX_PER_GROUP", "0"))  # cap per (subset, label, split); 0 = keep all
AUTO_CAP_TOTAL = 60_000        # if the dataset is bigger than this, cap groups automatically
NORMALISE_FORMAT = True        # give PNG fakes a JPEG(q=96) history -> kills the PNG-vs-JPEG shortcut
VAL_FRACTION = 0.2             # only used when the dataset has no train/val folders
TRAIN_GENERATOR = "auto"       # generator whose fakes are "seen" in training (auto: sd14 > sd15 > ...)
RUN_GEOMETRY, RUN_CLIP = True, True
N_CPU = os.cpu_count() or 2
CHUNK = 2000                   # images per resumable chunk
SEED = 0

FEAT_DIR = OUT_DIR / "features"
for d in (OUT_DIR, FEAT_DIR, OUT_DIR / "physfeat"):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(OUT_DIR))
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"data roots: {DATA_ROOTS}\noutput dir: {OUT_DIR}\nCPUs: {N_CPU}")

## 1 · Feature-extraction package
Written to `/kaggle/working/physfeat/` so later notebooks can import it from this notebook's output.

In [ ]:
%%writefile {OUT_DIR}/physfeat/__init__.py
"""physfeat — physics-based features for AI-generated image detection."""

In [ ]:
%%writefile {OUT_DIR}/physfeat/imaging.py
"""Image I/O shared by every feature extractor.

* `read_meta`      – format, size, JPEG quality, hash (for the bias audit / dedupe)
* `load_rgb`       – decode an image the same way for every extractor, optionally
                     neutralising the PNG-vs-JPEG shortcut and applying a
                     robustness perturbation (JPEG re-compression, resizing, blur).
"""
from __future__ import annotations

import hashlib
import io

import numpy as np
from PIL import Image, ImageFile, ImageFilter

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff", ".jfif"}
NORMALISE_JPEG_Q = 96  # GenImage de-biasing protocol (Grommelt et al., 2024)

# ----------------------------------------------------------------------------
# JPEG quality estimation (compare the luminance table with Pillow's own tables)
# ----------------------------------------------------------------------------
_QREF = None


def _qref():
    global _QREF
    if _QREF is None:
        tiny = Image.new("RGB", (16, 16), (128, 128, 128))
        ref = {}
        for q in range(1, 101):
            buf = io.BytesIO()
            tiny.save(buf, "JPEG", quality=q)
            ref[q] = np.array(Image.open(io.BytesIO(buf.getvalue())).quantization[0], float)
        _QREF = ref
    return _QREF


def estimate_jpeg_quality(im: Image.Image) -> float:
    q = getattr(im, "quantization", None)
    if not q or 0 not in q:
        return float("nan")
    t = np.array(q[0], float)
    ref = _qref()
    return float(min(ref, key=lambda k: np.abs(ref[k] - t).mean()))


def file_md5(path, chunk=1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def read_meta(path) -> dict:
    out = {"fmt": None, "width": np.nan, "height": np.nan, "jpeg_q": np.nan,
           "bytes": np.nan, "md5": None, "meta_error": None}
    try:
        import os
        out["bytes"] = os.path.getsize(path)
        out["md5"] = file_md5(path)
        with Image.open(path) as im:
            out["fmt"] = im.format
            out["width"], out["height"] = im.size
            if im.format == "JPEG":
                out["jpeg_q"] = estimate_jpeg_quality(im)
    except Exception as e:  # corrupt / unreadable
        out["meta_error"] = repr(e)[:200]
    return out


# ----------------------------------------------------------------------------
# decoding + perturbations
# ----------------------------------------------------------------------------
def jpeg_roundtrip(im: Image.Image, quality: int) -> Image.Image:
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=quality)
    return Image.open(io.BytesIO(buf.getvalue())).convert("RGB")


def _resize(im, s):
    w, h = im.size
    return im.resize((max(8, round(w * s)), max(8, round(h * s))), Image.BICUBIC)


PERTURBATIONS = {
    None: lambda im: im,
    "jpeg90": lambda im: jpeg_roundtrip(im, 90),
    "jpeg75": lambda im: jpeg_roundtrip(im, 75),
    "jpeg50": lambda im: jpeg_roundtrip(im, 50),
    "down0.5": lambda im: _resize(im, 0.5),
    "blur1": lambda im: im.filter(ImageFilter.GaussianBlur(1.0)),
}


def load_pil(path, normalise_format: bool = True, perturb: str | None = None) -> Image.Image:
    with Image.open(path) as im:
        fmt = im.format
        im = im.convert("RGB")
    if normalise_format and fmt != "JPEG":
        # PNG fakes vs JPEG reals is a known dataset shortcut: give every image a
        # JPEG history so the detector cannot key on the container format.
        im = jpeg_roundtrip(im, NORMALISE_JPEG_Q)
    return PERTURBATIONS[perturb](im)


def load_rgb(path, normalise_format: bool = True, perturb: str | None = None) -> np.ndarray:
    return np.asarray(load_pil(path, normalise_format, perturb))


def stable_seed(path) -> int:
    return int(hashlib.md5(str(path).encode()).hexdigest()[:8], 16)

In [ ]:
%%writefile {OUT_DIR}/physfeat/geometry.py
"""
Projective-geometry (perspective) physics features for AI-generated image detection.

Idea
----
In a real photograph taken with a (roughly) pinhole camera, images of parallel 3-D
lines converge *exactly* to a common vanishing point (VP), and the VPs of mutually
orthogonal directions obey camera constraints (principal point near the image
centre, positive focal length). Generators learn perspective only statistically,
so their lines tend to *almost* converge: scattered intersections, one VP split
into several nearby ones, and VP triplets that imply an impossible camera.

Pipeline
--------
1. Resize (max side = 512 by default) -> grayscale -> LSD line segments.
2. Sequential, length-weighted RANSAC to find up to 3 dominant VPs
   (VPs may be finite or at infinity; everything is done in homogeneous coords).
3. Least-squares refinement of each VP, then measurement of how well its
   inlier segments actually converge (angular residuals).
4. Camera-consistency checks between VPs (implied focal length, principal point).

All coordinates are normalised: origin at the image centre, unit = half diagonal.
Features are NaN when they are undefined (e.g. no lines in a photo of a cat);
downstream models must handle NaN (HistGradientBoosting does natively).
"""
from __future__ import annotations

import hashlib
from dataclasses import dataclass, field

import cv2
import numpy as np

# ----------------------------------------------------------------------------
# configuration
# ----------------------------------------------------------------------------
MAX_SIDE = 512            # images are resized so the longer side is this many px
MIN_LEN_FRAC = 0.025      # keep segments longer than this fraction of the diagonal
LONG_LEN_FRAC = 0.08      # "long" segments (used for the unexplained-lines feature)
INLIER_DEG = 1.5          # a segment supports a VP if its angular error is below this
TIGHT_DEG = 0.5           # threshold for the "tight convergence" fraction
N_HYP = 400               # RANSAC hypotheses per VP
MAX_VPS = 3
MIN_VP_SUPPORT = 4        # minimum number of inlier segments for a VP
MIN_VP_LEN_FRAC = 0.04    # minimum share of total line length for a VP
MAX_SEGMENTS = 1500       # cap (longest kept) to bound runtime on very busy images
FINITE_LIMIT = 8.0        # |VP| beyond this many half-diagonals is treated as "at infinity"

FEATURE_NAMES = [
    "geo_log_nseg",          # log(1 + number of segments kept)
    "geo_line_density",      # total segment length / image diagonal
    "geo_n_vps",             # number of VPs found (0..3)
    "geo_frac_vp1", "geo_frac_vp2", "geo_frac_vp3",   # share of line length supporting each VP
    "geo_frac_long_unexpl",  # share of long-segment length explained by no VP
    "geo_res_vp1", "geo_res_vp2", "geo_res_vp3",      # length-weighted mean angular residual (deg)
    "geo_tight_vp1", "geo_tight_vp2", "geo_tight_vp3",  # share of inlier length with residual < TIGHT_DEG
    "geo_cv_res_vp1",        # split-half cross-validated residual of VP1 (deg)
    "geo_min_vp_sep_deg",    # smallest angle between VP directions (split VPs are suspicious)
    "geo_ortho_cos_min",     # best |cos| between VP directions under the assumed camera
    "geo_f2_12",             # signed log focal^2 implied by VP1-VP2 orthogonality
    "geo_pp_offset",         # principal point (orthocentre of 3 finite VPs) distance from centre
]


@dataclass
class GeometryResult:
    features: dict
    segments: np.ndarray = field(default=None, repr=False)   # (N,4) pixel coords in resized image
    labels: np.ndarray = field(default=None, repr=False)     # (N,) VP index or -1
    residuals: np.ndarray = field(default=None, repr=False)  # (N,) angular error to own VP (deg)
    vps_px: list = field(default_factory=list)                # VPs in resized-pixel homogeneous coords
    image_rgb: np.ndarray = field(default=None, repr=False)  # resized RGB image (for plotting)


# ----------------------------------------------------------------------------
# image loading
# ----------------------------------------------------------------------------
def load_image(path_or_array, max_side: int = MAX_SIDE) -> np.ndarray:
    """Return an RGB uint8 image whose longer side is `max_side`."""
    if isinstance(path_or_array, np.ndarray):
        rgb = path_or_array
    else:
        bgr = cv2.imread(str(path_or_array), cv2.IMREAD_COLOR)
        if bgr is None:  # some formats (e.g. certain webp/gif) -> fall back to PIL
            from PIL import Image
            rgb = np.array(Image.open(path_or_array).convert("RGB"))
        else:
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if rgb.ndim == 2:
        rgb = np.stack([rgb] * 3, -1)
    h, w = rgb.shape[:2]
    s = max_side / max(h, w)
    if abs(s - 1) > 1e-3:
        interp = cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC
        rgb = cv2.resize(rgb, (round(w * s), round(h * s)), interpolation=interp)
    return rgb


# ----------------------------------------------------------------------------
# core geometry helpers
# ----------------------------------------------------------------------------
_LSD = None


def _lsd():
    global _LSD
    if _LSD is None:  # one detector per process
        _LSD = cv2.createLineSegmentDetector(cv2.LSD_REFINE_STD)
    return _LSD


def detect_segments(gray: np.ndarray) -> np.ndarray:
    h, w = gray.shape
    diag = np.hypot(h, w)
    lines = _lsd().detect(gray)[0]
    if lines is None:
        return np.zeros((0, 4), np.float64)
    segs = lines.reshape(-1, 4).astype(np.float64)
    L = np.hypot(segs[:, 2] - segs[:, 0], segs[:, 3] - segs[:, 1])
    keep = L >= MIN_LEN_FRAC * diag
    segs, L = segs[keep], L[keep]
    if len(segs) > MAX_SEGMENTS:
        segs = segs[np.argsort(-L)[:MAX_SEGMENTS]]
    return segs


def _normalise(segs, h, w):
    c = np.array([w / 2.0, h / 2.0])
    s = np.hypot(h, w) / 2.0
    p1 = (segs[:, :2] - c) / s
    p2 = (segs[:, 2:] - c) / s
    return p1, p2, c, s


def _angles_to_vps(mid, d, vps):
    """Angular error (deg, 0..90) between each segment and the line joining its
    midpoint to each VP.  vps: (K,3) homogeneous.  Returns (K,N)."""
    # direction from midpoint towards VP; valid for finite and infinite VPs
    dv = vps[:, None, :2] - mid[None] * vps[:, None, 2:3]          # (K,N,2)
    num = np.abs(dv[..., 0] * d[None, :, 0] + dv[..., 1] * d[None, :, 1])
    den = np.linalg.norm(dv, axis=-1) * np.linalg.norm(d, axis=-1)[None] + 1e-12
    return np.degrees(np.arccos(np.clip(num / den, 0.0, 1.0)))


def _refine_vp(lines_n, weights):
    """Weighted least squares VP: argmin_{|v|=1} sum w_i (l_i . v)^2."""
    A = lines_n * np.sqrt(weights)[:, None]
    _, _, vt = np.linalg.svd(A, full_matrices=False)
    v = vt[-1]
    return v / (np.linalg.norm(v) + 1e-12)


def _wmean(x, w):
    return float(np.sum(x * w) / (np.sum(w) + 1e-12))


def _is_finite(v):
    return abs(v[2]) > 1e-9 and np.linalg.norm(v[:2] / v[2]) < FINITE_LIMIT


def _ray(v):
    """3-D direction of a VP under a camera with principal point at the centre and
    focal length = 1 half-diagonal (~53 deg diagonal FOV); used only for
    separation angles, not for the camera-consistency tests."""
    r = np.array([v[0], v[1], v[2]], float)
    return r / (np.linalg.norm(r) + 1e-12)


# ----------------------------------------------------------------------------
# main entry point
# ----------------------------------------------------------------------------
def extract(path_or_array, seed: int | None = None, keep_debug: bool = False) -> GeometryResult:
    rgb = load_image(path_or_array)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    diag = np.hypot(h, w)
    feats = {k: np.nan for k in FEATURE_NAMES}

    segs = detect_segments(gray)
    n = len(segs)
    feats["geo_log_nseg"] = float(np.log1p(n))
    L = np.hypot(segs[:, 2] - segs[:, 0], segs[:, 3] - segs[:, 1]) if n else np.zeros(0)
    feats["geo_line_density"] = float(L.sum() / diag)
    feats["geo_n_vps"] = 0.0
    labels = -np.ones(n, int)
    resid = np.full(n, np.nan)
    vps = []

    if n >= 2 * MIN_VP_SUPPORT:
        if seed is None:
            seed = int(hashlib.md5(str(path_or_array if not isinstance(path_or_array, np.ndarray)
                                        else path_or_array.shape).encode()).hexdigest()[:8], 16)
        rng = np.random.default_rng(seed)
        p1, p2, _, _ = _normalise(segs, h, w)
        mid, d = (p1 + p2) / 2, p2 - p1
        Ln = np.linalg.norm(d, axis=1)
        lines = np.cross(np.c_[p1, np.ones(n)], np.c_[p2, np.ones(n)])
        lines /= np.linalg.norm(lines[:, :2], axis=1, keepdims=True) + 1e-12  # (a,b) unit
        total_len = Ln.sum()
        free = np.ones(n, bool)

        for k in range(MAX_VPS):
            idx = np.flatnonzero(free)
            if len(idx) < MIN_VP_SUPPORT:
                break
            prob = Ln[idx] / Ln[idx].sum()
            i = rng.choice(idx, N_HYP, p=prob)
            j = rng.choice(idx, N_HYP, p=prob)
            ok = i != j
            hyp = np.cross(lines[i[ok]], lines[j[ok]])
            nrm = np.linalg.norm(hyp, axis=1)
            hyp = hyp[nrm > 1e-9] / nrm[nrm > 1e-9, None]
            if len(hyp) == 0:
                break
            ang = _angles_to_vps(mid[idx], d[idx], hyp)          # (H, |idx|)
            score = ((ang < INLIER_DEG) * Ln[idx][None]).sum(1)
            v = hyp[np.argmax(score)]
            # refine twice (inliers -> LS fit -> inliers)
            for _ in range(2):
                a = _angles_to_vps(mid[idx], d[idx], v[None])[0]
                inl = idx[a < INLIER_DEG]
                if len(inl) < 2:
                    break
                v = _refine_vp(lines[inl], Ln[inl])
            a_all = _angles_to_vps(mid, d, v[None])[0]
            inl = idx[a_all[idx] < INLIER_DEG]
            if len(inl) < MIN_VP_SUPPORT or Ln[inl].sum() < MIN_VP_LEN_FRAC * total_len:
                break
            labels[inl] = k
            resid[inl] = a_all[inl]
            free[inl] = False
            vps.append(v)

            kk = k + 1
            feats[f"geo_frac_vp{kk}"] = float(Ln[inl].sum() / total_len)
            feats[f"geo_res_vp{kk}"] = _wmean(a_all[inl], Ln[inl])
            feats[f"geo_tight_vp{kk}"] = float(Ln[inl][a_all[inl] < TIGHT_DEG].sum() / Ln[inl].sum())
            if k == 0 and len(inl) >= 2 * MIN_VP_SUPPORT:
                # split-half cross-validation: fit on one half, measure the other
                perm = rng.permutation(inl)
                A, B = perm[: len(perm) // 2], perm[len(perm) // 2:]
                vA, vB = _refine_vp(lines[A], Ln[A]), _refine_vp(lines[B], Ln[B])
                eAB = _wmean(_angles_to_vps(mid[B], d[B], vA[None])[0], Ln[B])
                eBA = _wmean(_angles_to_vps(mid[A], d[A], vB[None])[0], Ln[A])
                feats["geo_cv_res_vp1"] = 0.5 * (eAB + eBA)

        feats["geo_n_vps"] = float(len(vps))
        long_ = Ln >= (LONG_LEN_FRAC * diag) / (diag / 2)
        if long_.any():
            feats["geo_frac_long_unexpl"] = float(Ln[long_ & (labels < 0)].sum() / Ln[long_].sum())

        # ---- relations between VPs --------------------------------------------------
        if len(vps) >= 2:
            rays = [_ray(v) for v in vps]
            seps, coss = [], []
            for a in range(len(vps)):
                for b in range(a + 1, len(vps)):
                    c = abs(float(np.dot(rays[a], rays[b])))
                    seps.append(np.degrees(np.arccos(min(1.0, c))))
                    coss.append(c)
            feats["geo_min_vp_sep_deg"] = float(min(seps))
            feats["geo_ortho_cos_min"] = float(min(coss))

            v1, v2 = vps[0], vps[1]
            if _is_finite(v1) and _is_finite(v2):
                # orthogonality with principal point at centre:
                # f^2 = -(x1 x2 + y1 y2) for VPs in normalised inhomogeneous coords
                q1, q2 = v1[:2] / v1[2], v2[:2] / v2[2]
                f2 = -float(np.dot(q1, q2))
                feats["geo_f2_12"] = float(np.sign(f2) * np.log1p(abs(f2)))

        if len(vps) == 3 and all(_is_finite(v) for v in vps):
            # principal point = orthocentre of the VP triangle (3 orthogonal directions)
            q = [v[:2] / v[2] for v in vps]
            A = np.array([q[2] - q[1], q[2] - q[0]])
            b = np.array([np.dot(q[0], q[2] - q[1]), np.dot(q[1], q[2] - q[0])])
            if abs(np.linalg.det(A)) > 1e-9:
                pp = np.linalg.solve(A, b)
                feats["geo_pp_offset"] = float(min(np.linalg.norm(pp), 10.0))

    res = GeometryResult(features=feats)
    if keep_debug:
        c = np.array([w / 2.0, h / 2.0]); s = diag / 2.0
        T = np.array([[s, 0, c[0]], [0, s, c[1]], [0, 0, 1.0]])   # normalised -> pixel
        res.segments, res.labels, res.residuals = segs, labels, resid
        res.vps_px = [T @ v for v in vps]
        res.image_rgb = rgb
    return res


def extract_features(path) -> dict:
    """Picklable wrapper for multiprocessing pools; never raises."""
    try:
        return extract(path).features
    except Exception as e:  # corrupt file etc.
        out = {k: np.nan for k in FEATURE_NAMES}
        out["geo_error"] = repr(e)[:200]
        return out


# ----------------------------------------------------------------------------
# visualisation (used by notebooks and the demo)
# ----------------------------------------------------------------------------
VP_COLOURS = [(230, 57, 70), (29, 120, 220), (46, 170, 90)]


def draw_evidence(res: GeometryResult, show_vp_rays: bool = True) -> np.ndarray:
    """Overlay: segments coloured by VP, thickness/alpha by convergence error;
    unexplained long lines in yellow; short rays towards each finite VP."""
    img = res.image_rgb.copy()
    over = img.copy()
    h, w = img.shape[:2]
    diag = np.hypot(h, w)
    for (x1, y1, x2, y2), lab, r in zip(res.segments, res.labels, res.residuals):
        p, q = (int(x1), int(y1)), (int(x2), int(y2))
        if lab >= 0:
            col = VP_COLOURS[lab % 3]
            cv2.line(over, p, q, col, 2, cv2.LINE_AA)
        elif np.hypot(x2 - x1, y2 - y1) >= LONG_LEN_FRAC * diag:
            cv2.line(over, p, q, (255, 200, 0), 2, cv2.LINE_AA)
    if show_vp_rays:
        for k, v in enumerate(res.vps_px):
            if abs(v[2]) < 1e-9:
                continue
            vx, vy = v[0] / v[2], v[1] / v[2]
            sel = np.flatnonzero(res.labels == k)
            for s in sel[np.argsort(-np.hypot(res.segments[sel, 2] - res.segments[sel, 0],
                                             res.segments[sel, 3] - res.segments[sel, 1]))[:12]]:
                mx, my = (res.segments[s, :2] + res.segments[s, 2:]) / 2
                # extend the segment's midpoint toward the VP (clipped to a long ray)
                dx, dy = vx - mx, vy - my
                t = min(1.0, 0.6 * diag / (np.hypot(dx, dy) + 1e-9))
                cv2.line(over, (int(mx), int(my)), (int(mx + t * dx), int(my + t * dy)),
                         VP_COLOURS[k % 3], 1, cv2.LINE_AA)
            if 0 <= vx < w and 0 <= vy < h:
                cv2.circle(over, (int(vx), int(vy)), 6, VP_COLOURS[k % 3], -1, cv2.LINE_AA)
    return cv2.addWeighted(over, 0.85, img, 0.15, 0)

In [ ]:
%%writefile {OUT_DIR}/physfeat/dataset.py
"""Find GenImage-style images on disk and build a labelled manifest.

Works with the original GenImage layout
    imagenet_ai_0419_sdv4/train/ai/*.png, .../val/nature/*.JPEG
and with the common Kaggle re-uploads (tiny-genimage, unbiased-tiny-genimage),
which use folder names like  midjourney/ai, sd14/nature, real/, fake/ ...
Labels, generator and split are inferred from path components.
"""
from __future__ import annotations

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

from .imaging import IMG_EXTS

REAL_TOKENS = {"nature", "real", "0real", "natural", "imagenet", "authentic", "genuine", "0"}
FAKE_TOKENS = {"ai", "fake", "1fake", "generated", "synthetic", "aigenerated", "1"}
SPLIT_TOKENS = {"train": "train", "training": "train", "val": "val", "valid": "val",
                "validation": "val", "test": "test", "testing": "test"}

# normalised alias -> canonical generator name (checked as substrings, longest first)
GEN_ALIASES = {
    "midjourney": "midjourney", "imagenetmidjourney": "midjourney", "mj": "midjourney",
    "stablediffusionv14": "sd14", "stablediffusionv4": "sd14", "sdv14": "sd14", "sd14": "sd14",
    "sdv4": "sd14",
    "stablediffusionv15": "sd15", "stablediffusionv5": "sd15", "sdv15": "sd15", "sd15": "sd15",
    "sdv5": "sd15",
    "adm": "adm", "glide": "glide", "wukong": "wukong", "vqdm": "vqdm", "biggan": "biggan",
    "sdxl": "sdxl", "flux": "flux", "dalle": "dalle",
    "stablediffusion": "sd", "sd": "sd",   # version-less fallbacks (checked after the specific ones)
}
_ALIAS_ORDER = sorted(GEN_ALIASES, key=len, reverse=True)
_SHORT = {"mj", "adm", "sd"}  # short aliases must match a whole component


def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", s.lower())


def parse_path(path: str) -> dict:
    parts = [_norm(p) for p in Path(path).parts[:-1]]
    label, gen, split = None, None, None
    for p in reversed(parts):               # innermost folder wins
        if label is None and p in REAL_TOKENS:
            label = 0
        elif label is None and p in FAKE_TOKENS:
            label = 1
        if split is None and p in SPLIT_TOKENS:
            split = SPLIT_TOKENS[p]
        if gen is None:
            for a in _ALIAS_ORDER:
                if (a in _SHORT and (p == a or p.endswith(a) and p.startswith("imagenet"))) or \
                   (a not in _SHORT and a in p):
                    gen = GEN_ALIASES[a]
                    break
    if label is None and gen is not None:
        # flat layouts (e.g. unbiased-tiny-genimage: BigGAN/, ADM/, ..., plus a real folder):
        # a generator-named folder with no real/fake sub-folder holds that generator's fakes
        label = 1
    return {"label": label, "subset": gen, "split": split}


def scan(roots, exts=IMG_EXTS) -> pd.DataFrame:
    rows = []
    for root in roots:
        for dp, _, fns in os.walk(root, followlinks=True):
            for fn in fns:
                if os.path.splitext(fn)[1].lower() in exts:
                    p = os.path.join(dp, fn)
                    rows.append({"path": p, **parse_path(os.path.relpath(p, root))})
    df = pd.DataFrame(rows, columns=["path", "label", "subset", "split"])
    # fakes are attributed to the generator of their subset; reals are just "real"
    df["generator"] = np.where(df["label"] == 0, "real", df["subset"])
    return df


def folder_tree(root, max_depth=4, max_children=12) -> str:
    """Compact tree with image counts, to show what the scanner is looking at."""
    lines = []

    def count(d):
        n = 0
        for _, _, fns in os.walk(d):
            n += sum(os.path.splitext(f)[1].lower() in IMG_EXTS for f in fns)
        return n

    def rec(d, depth):
        subs = sorted([e for e in os.scandir(d) if e.is_dir()], key=lambda e: e.name)
        for e in subs[:max_children]:
            lines.append("  " * depth + f"{e.name}/  ({count(e.path)} imgs)")
            if depth + 1 < max_depth:
                rec(e.path, depth + 1)
        if len(subs) > max_children:
            lines.append("  " * depth + f"... {len(subs) - max_children} more")
    lines.append(f"{root}/  ({count(root)} imgs)")
    rec(root, 1)
    return "\n".join(lines)

In [ ]:
%%writefile {OUT_DIR}/physfeat/protocol.py
"""Cross-generator evaluation protocol.

Train:  real images (train split)  vs  fakes from ONE generator (train split)
Test:   for every generator g:  real images (val split)  vs  fakes from g (val split)

The generator used for training is "seen"; every other generator is "unseen".
Generalisation = performance on the unseen ones.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

PREFERRED_TRAIN_GENERATORS = ["sd14", "sd15", "sd", "adm", "glide", "vqdm", "wukong", "midjourney", "biggan"]


def pick_train_generator(df: pd.DataFrame, requested: str = "auto") -> str:
    gens = df.loc[(df.label == 1) & (df.split == "train"), "generator"].value_counts()
    if gens.empty:
        raise ValueError("No fake images in the train split — the folder scanner labelled nothing as fake. "
                         "Check the folder tree printed above and extend GEN_ALIASES / FAKE_TOKENS "
                         "in physfeat/dataset.py. Counts per generator/split:\n"
                         + df.groupby(["generator", "split"]).size().to_string())
    if requested != "auto":
        if requested not in gens.index:
            raise ValueError(f"{requested!r} has no training fakes; available: {list(gens.index)}")
        return requested
    for g in PREFERRED_TRAIN_GENERATORS:
        if gens.get(g, 0) >= 200:
            return g
    return gens.index[0]


def train_mask(df, train_gen):
    return (df.split == "train").values & ((df.label == 0) | (df.generator == train_gen)).values


def test_masks(df, min_fakes=5):
    """dict generator -> boolean mask over df (val reals + val fakes of that generator)."""
    val = (df.split == "val").values
    real = val & (df.label == 0).values
    out = {}
    for g in sorted(df.loc[df.label == 1, "generator"].dropna().unique()):
        m = real | (val & (df.generator == g).values)
        if (m & (df.label == 1).values).sum() >= min_fakes:
            out[g] = m
    return out


def evaluate(scores: dict, df: pd.DataFrame, train_gen: str, extra_mask=None) -> pd.DataFrame:
    """scores: model name -> array of P(fake) over the whole df. Returns a tidy table."""
    rows = []
    for g, m in test_masks(df).items():
        if extra_mask is not None:
            m = m & extra_mask
        y = df.label.values[m]
        if len(np.unique(y)) < 2:
            continue
        for name, s in scores.items():
            p = s[m]
            rows.append({"generator": g, "seen": g == train_gen, "model": name,
                         "auc": roc_auc_score(y, p),
                         "bal_acc": balanced_accuracy_score(y, p > 0.5),
                         "n_fake": int(y.sum()), "n_real": int((1 - y).sum())})
    return pd.DataFrame(rows)


def summary_table(res: pd.DataFrame, metric="auc") -> pd.DataFrame:
    if res.empty:
        return pd.DataFrame({"note": ["no generator had enough validation fakes to evaluate"]})
    t = res.pivot_table(index=["generator", "seen"], columns="model", values=metric).reset_index()
    t = t.sort_values(["seen", "generator"], ascending=[False, True]).set_index("generator")
    unseen = t[~t.seen].drop(columns="seen")
    t = t.drop(columns="seen")
    t.loc["MEAN (unseen)"] = unseen.mean()
    return t

In [ ]:
%%writefile {OUT_DIR}/physfeat/workers.py
"""Picklable per-image workers for multiprocessing pools."""
from __future__ import annotations

import numpy as np

from . import geometry
from .imaging import load_rgb, read_meta, stable_seed


def meta_worker(path):
    return read_meta(path)


def geometry_worker(args):
    """args = (path, normalise_format, perturb). Never raises."""
    path, normalise, perturb = args
    try:
        rgb = load_rgb(path, normalise, perturb)
        feats = geometry.extract(rgb, seed=stable_seed(path)).features
        feats["geo_error"] = None
    except Exception as e:
        feats = {k: np.nan for k in geometry.FEATURE_NAMES}
        feats["geo_error"] = repr(e)[:200]
    return feats

In [ ]:
%%writefile {OUT_DIR}/physfeat/clipfeat.py
"""CLIP ViT-L/14 image embeddings (the UnivFD-style data-driven baseline)."""
from __future__ import annotations

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

from .imaging import load_pil

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

PREPROCESS = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(CLIP_MEAN, CLIP_STD),
])


class ImageList(Dataset):
    def __init__(self, paths, normalise_format=True, perturb=None):
        self.paths, self.norm, self.perturb = list(paths), normalise_format, perturb

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            return PREPROCESS(load_pil(self.paths[i], self.norm, self.perturb)), True
        except Exception:
            return torch.zeros(3, 224, 224), False


def load_clip(model_name: str, device: str):
    from transformers import CLIPVisionModelWithProjection
    dtype = torch.float16 if device.startswith("cuda") else torch.float32
    model = CLIPVisionModelWithProjection.from_pretrained(model_name, torch_dtype=dtype)
    return model.to(device).eval()


@torch.no_grad()
def embed(model, paths, device, batch_size=128, num_workers=4,
          normalise_format=True, perturb=None, progress=True):
    """Returns (N, D) float16 L2-normalised embeddings and an (N,) ok-mask."""
    dl = DataLoader(ImageList(paths, normalise_format, perturb), batch_size=batch_size,
                    num_workers=num_workers, pin_memory=device.startswith("cuda"))
    dtype = next(model.parameters()).dtype
    feats, oks = [], []
    it = dl
    if progress:
        from tqdm.auto import tqdm
        it = tqdm(dl, desc="CLIP", leave=False)
    for x, ok in it:
        e = model(pixel_values=x.to(device, dtype)).image_embeds.float()
        e = torch.nn.functional.normalize(e, dim=-1)
        feats.append(e.cpu().numpy().astype(np.float16))
        oks.append(ok.numpy())
    return np.concatenate(feats), np.concatenate(oks)

In [ ]:
import importlib, numpy as np, pandas as pd
from tqdm.auto import tqdm
from multiprocessing import Pool
import physfeat
from physfeat import dataset as D, geometry as G, protocol as P, workers as W
for m in (D, G, P, W):
    importlib.reload(m)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

## 2 · Find the images
Labels, generator and split are inferred from folder names. Check the tree and the table below — if anything shows up as unlabelled, adjust `REAL_TOKENS` / `GEN_ALIASES` in `physfeat/dataset.py`.

In [ ]:
for r in DATA_ROOTS:
    print(D.folder_tree(r, max_depth=4), "\n")

raw = pd.concat([D.scan([r]) for r in DATA_ROOTS], ignore_index=True)
print(f"{len(raw):,} image files found")
unlabelled = raw[raw.label.isna() | ((raw.label == 1) & raw.subset.isna())]
if len(unlabelled):
    print(f"WARNING: {len(unlabelled):,} files have no real/fake label or no generator and are dropped. By folder:")
    print(unlabelled.path.map(lambda p: os.path.basename(os.path.dirname(p))).value_counts().head(15).to_string())
df = raw.drop(unlabelled.index).copy()
df["label"] = df.label.astype(int)
df["subset"] = df.subset.fillna("shared")
df["split"] = df.split.fillna("none").replace({"test": "val"})
assert len(df), "no labelled images found — check DATA_ROOTS and the folder names above"
pd.crosstab([df.generator], df.split, margins=True)

## 3 · Sample, metadata, de-duplication, splits

In [ ]:
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
cap = MAX_PER_GROUP
if cap == 0 and len(df) > AUTO_CAP_TOTAL:
    cap = int(np.ceil(AUTO_CAP_TOTAL / df.groupby(["subset", "label", "split"]).ngroups))
    print(f"dataset is large ({len(df):,}); capping each (subset,label,split) group at {cap}")
if cap > 0:
    df = df[df.groupby(["subset", "label", "split"]).cumcount() < cap].reset_index(drop=True)

t0 = time.time()
with Pool(N_CPU) as pool:
    meta = list(tqdm(pool.imap(W.meta_worker, df.path.tolist(), chunksize=64), total=len(df), desc="metadata"))
df = pd.concat([df, pd.DataFrame(meta)], axis=1)
print(f"metadata: {time.time()-t0:.0f}s")

bad = df.meta_error.notna()
print(f"unreadable files dropped: {bad.sum()}")
df = df[~bad].drop(columns="meta_error")

# exact duplicates (same bytes) -> keep one copy, preferring the train split, to avoid train/test leakage
n0 = len(df)
df = df.assign(_o=df.split.map({"train": 0, "none": 1, "val": 2})).sort_values(["_o", "path"])
df = df.drop_duplicates("md5").drop(columns="_o")
print(f"exact duplicates removed: {n0 - len(df)}")

# assign a split where the dataset has none (stratified by generator)
rng = np.random.default_rng(SEED)
none = df.split == "none"
if none.any():
    df.loc[none, "split"] = np.where(rng.random(none.sum()) < VAL_FRACTION, "val", "train")

df["bpp"] = df["bytes"] * 8 / (df.width * df.height)       # bits per pixel (compression strength)
df = df.sort_values("path").reset_index(drop=True)
df.to_csv(OUT_DIR / "manifest.csv", index=False)
TRAIN_GEN = P.pick_train_generator(df, TRAIN_GENERATOR)
print(f"\nmanifest: {len(df):,} images  ·  training ('seen') generator: {TRAIN_GEN}")
pd.crosstab(df.generator, df.split, margins=True)

## 4 · Bias audit — can *metadata alone* tell real from fake?
Known trap (Grommelt et al., *Fake or JPEG?*, 2024): in GenImage the fakes are PNG and the reals are JPEG with varied sizes, so a detector can learn file format instead of fakeness.
We train a classifier on **metadata only** (format, JPEG quality, size, bits-per-pixel) with the same cross-generator protocol. AUC near 0.5 = clean dataset. AUC near 1 = your results would be meaningless without the fix.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

audit = df.groupby("generator").agg(
    n=("path", "size"), jpeg_share=("fmt", lambda s: (s == "JPEG").mean()),
    jpeg_q_median=("jpeg_q", "median"), width_median=("width", "median"),
    height_median=("height", "median"), bits_per_pixel=("bpp", "median")).round(3)
display(audit)

meta_X = pd.DataFrame({
    "is_jpeg": (df.fmt == "JPEG").astype(float), "jpeg_q": df.jpeg_q,
    "width": df.width, "height": df.height, "aspect": df.width / df.height, "bpp": df.bpp}).values
tr = P.train_mask(df, TRAIN_GEN)
meta_clf = HistGradientBoostingClassifier(max_iter=200, class_weight="balanced", random_state=SEED)
meta_clf.fit(meta_X[tr], df.label.values[tr])
meta_res = P.evaluate({"metadata_only": meta_clf.predict_proba(meta_X)[:, 1]}, df, TRAIN_GEN)
display(P.summary_table(meta_res).round(3))
meta_auc = float(meta_res.loc[~meta_res.seen, "auc"].mean()) if len(meta_res) else float("nan")
if meta_auc > 0.75:
    print(f"WARNING: metadata alone reaches AUC {meta_auc:.2f} on unseen generators. The dataset is biased.\n"
          "NORMALISE_FORMAT removes the format cue from the features; image-size / JPEG-quality cues can still\n"
          "leak through resizing. Prefer the *unbiased* tiny GenImage and report this number in your write-up.")
else:
    print(f"OK: metadata-only AUC on unseen generators = {meta_auc:.2f} (little shortcut signal).")

## 5 · Physics features: projective geometry (CPU)
Line segments → up to 3 vanishing points → how exactly the lines converge and whether the implied camera is physically possible. Chunked and resumable.

In [ ]:
def chunked(name, fn, ext):
    # Run fn(paths)->DataFrame|ndarray over CHUNK-sized slices; reuse chunks whose path list matches.
    paths = df.path.tolist()
    outs = []
    for ci, s in enumerate(range(0, len(paths), CHUNK)):
        ps = paths[s:s + CHUNK]
        f, fp = FEAT_DIR / f"{name}_{ci:04d}.{ext}", FEAT_DIR / f"{name}_{ci:04d}.paths.txt"
        if f.exists() and fp.exists() and fp.read_text().splitlines() == ps:
            outs.append(pd.read_parquet(f) if ext == "parquet" else np.load(f)); continue
        t0 = time.time(); out = fn(ps)
        (out.to_parquet(f) if ext == "parquet" else np.save(f, out)); fp.write_text("\n".join(ps))
        outs.append(out)
        print(f"{name} chunk {ci}: {len(ps)} imgs in {time.time()-t0:.0f}s")
    return outs

def geometry_fn(ps):
    with Pool(N_CPU) as pool:
        rows = list(tqdm(pool.imap(W.geometry_worker, [(p, NORMALISE_FORMAT, None) for p in ps], chunksize=8),
                         total=len(ps), desc="geometry", leave=False))
    out = pd.DataFrame(rows)
    out.insert(0, "path", ps)
    out["geo_error"] = out["geo_error"].astype("string")
    return out

if RUN_GEOMETRY:
    geo = pd.concat(chunked("geo", geometry_fn, "parquet"), ignore_index=True)
    assert (geo.path.values == df.path.values).all()
    geo.to_parquet(FEAT_DIR / "geometry.parquet")
    print(f"geometry done · errors: {geo.geo_error.notna().sum()} · "
          f"images with >=2 vanishing points: {(geo.geo_n_vps >= 2).mean():.0%}")
    display(geo[G.FEATURE_NAMES].describe().T.round(3))

## 6 · Baseline features: CLIP ViT-L/14 (GPU)

In [ ]:
if RUN_CLIP:
    import torch
    from physfeat import clipfeat as C
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("WARNING: no GPU — CLIP ViT-L/14 on CPU is very slow. Turn on the T4 accelerator.")
    clip_model = C.load_clip(CLIP_MODEL, device)
    bs = 128 if device == "cuda" else 16

    def clip_fn(ps):
        emb, ok = C.embed(clip_model, ps, device, batch_size=bs, num_workers=N_CPU,
                          normalise_format=NORMALISE_FORMAT)
        if (~ok).any():
            print(f"  {(~ok).sum()} images failed to load (zero vectors)")
        return emb

    clip = np.concatenate(chunked("clip", clip_fn, "npy"))
    assert len(clip) == len(df)
    np.save(FEAT_DIR / "clip_vitl14.npy", clip)
    print("CLIP embeddings:", clip.shape, clip.dtype)
    del clip_model
    if device == "cuda":
        torch.cuda.empty_cache()

## 7 · First look at the result (preview)
Quick models — the proper training, fusion and robustness study are in notebook 03.

* **CLIP (baseline)** — logistic regression on CLIP embeddings (UnivFD-style linear probe)
* **Physics (geometry)** — gradient boosting on the 18 geometry features (handles "no lines" = NaN)
* **Fused** — stacking: a logistic regression on the two models' out-of-fold scores

Trained on `real` vs **one** generator, tested on every generator. The row that matters is **MEAN (unseen)**.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold

manifest = pd.read_csv(OUT_DIR / "manifest.csv")
geo = pd.read_parquet(FEAT_DIR / "geometry.parquet")
clip = np.load(FEAT_DIR / "clip_vitl14.npy").astype(np.float32)
y = manifest.label.values
tr = P.train_mask(manifest, TRAIN_GEN)
Xg = geo[G.FEATURE_NAMES].values
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

clip_clf = LogisticRegression(C=2.0, max_iter=3000, class_weight="balanced")
geo_clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced",
                                         random_state=SEED)
oof_c = cross_val_predict(clip_clf, clip[tr], y[tr], cv=cv, method="predict_proba")[:, 1]
oof_g = cross_val_predict(geo_clf, Xg[tr], y[tr], cv=cv, method="predict_proba")[:, 1]
clip_clf.fit(clip[tr], y[tr]); geo_clf.fit(Xg[tr], y[tr])
s_c, s_g = clip_clf.predict_proba(clip)[:, 1], geo_clf.predict_proba(Xg)[:, 1]

logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / np.clip(1 - p, 1e-6, 1))
stack = LogisticRegression(class_weight="balanced").fit(np.c_[logit(oof_c), logit(oof_g)], y[tr])
s_f = stack.predict_proba(np.c_[logit(s_c), logit(s_g)])[:, 1]
print("stacking weights (CLIP, geometry):", stack.coef_.round(2))

scores = {"CLIP (baseline)": s_c, "Physics: geometry": s_g, "Fused": s_f}
res_all = P.evaluate(scores, manifest, TRAIN_GEN)
print(f"\nAUC — all test images (seen generator = {TRAIN_GEN})")
display(P.summary_table(res_all).round(3))

lines = (geo.geo_n_vps >= 2).values
res_lines = P.evaluate(scores, manifest, TRAIN_GEN, extra_mask=lines)
print(f"\nAUC — only images with >= 2 vanishing points ({lines.mean():.0%} of images; where geometry can speak)")
display(P.summary_table(res_lines).round(3))

In [ ]:
import matplotlib.pyplot as plt
key = ["geo_res_vp1", "geo_tight_vp1", "geo_cv_res_vp1", "geo_min_vp_sep_deg", "geo_frac_long_unexpl", "geo_n_vps"]
sub = manifest.join(geo[G.FEATURE_NAMES])[lines]
order = ["real"] + sorted(g for g in sub.generator.unique() if g != "real")
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, k in zip(axes.ravel(), key):
    data = [sub.loc[sub.generator == g, k].dropna().values for g in order]
    ax.boxplot(data, showfliers=False)
    ax.set_xticks(range(1, len(order) + 1), order, rotation=45, ha="right")
    ax.set_title(k)
fig.suptitle("Geometry features by source (images with >= 2 vanishing points)")
fig.tight_layout(); fig.savefig(OUT_DIR / "geometry_boxplots.png", dpi=110); plt.show()

In [ ]:
# Evidence overlays: coloured lines = segments assigned to each vanishing point, yellow = long lines
# that converge nowhere. Real photos: tight bundles. Generated: split / scattered bundles.
from physfeat.imaging import load_rgb
def top_line_images(mask, n):
    cand = manifest[mask & lines].assign(nl=geo.geo_log_nseg[mask & lines])
    return cand.sort_values("nl", ascending=False).head(n)
val = (manifest.split == "val").values
picks = pd.concat([top_line_images(val & (y == 0), 3), top_line_images(val & (y == 1), 3)])
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (_, r) in zip(axes.ravel(), picks.iterrows()):
    res = G.extract(load_rgb(r.path, NORMALISE_FORMAT), keep_debug=True)
    f = res.features
    ax.imshow(G.draw_evidence(res)); ax.axis("off")
    ax.set_title(f"{r.generator} | VPs={int(f['geo_n_vps'])} res={f['geo_res_vp1']:.2f}° "
                 f"tight={f['geo_tight_vp1']:.2f}", fontsize=9)
fig.tight_layout(); fig.savefig(OUT_DIR / "geometry_evidence.png", dpi=110); plt.show()

## 8 · Save summary

In [ ]:
summary = {
    "n_images": int(len(manifest)),
    "counts": manifest.groupby(["generator", "split"]).size().unstack(fill_value=0).to_dict(),
    "train_generator": TRAIN_GEN,
    "normalise_format": NORMALISE_FORMAT,
    "metadata_only_unseen_auc": meta_auc,
    "preview_auc_all": P.summary_table(res_all).round(4).to_dict(),
    "preview_auc_lines": P.summary_table(res_lines).round(4).to_dict(),
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file() and "_0" not in p.name and "__pycache__" not in str(p):
        print(f"{p.stat().st_size/1e6:9.2f} MB  {p.relative_to(OUT_DIR)}")